# HadISD Data Download Notebook

This notebook will help you download a subset of the HadISD dataset directly from the Met Office website. The data will be stored in a user-specified directory (or a sensible default), and extracted for further processing (e.g., conversion to Zarr).

- **Source:** [HadISD v3.4.0.2023f](https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/download.html)
- **Instructions:**
    1. Set the download directory (or use the default).
    2. Download the data using Python's `requests` package.
    3. Extract the `.tar.gz` archive.
    4. The extracted files will be ready for use in the next notebook (`HadISD_to_zarr.ipynb`).

> **Note:** Download size is large. Ensure you have sufficient disk space and a stable internet connection.

In [1]:
import requests
from tqdm.auto import tqdm
import tarfile
import gzip
import shutil
from pathlib import Path

### Retrieve Path to Download Directory
The download location will default to a folder named "HadISD_data" in your home directory.<br>
If you want to change this, you can do so in the `Data_config.ipynb` configuration notebook. <br>


In [2]:
# ruff: noqa: F821
%run Data_Config.ipynb

### Download HadISD Data
The following code will download the HadISD data files. Some files take longer to download than others depending on time of day. To download different WMO datasets, you can change `wmo_id_ranges` in the `Data_Config.ipynb` notebook.

The full list of available data can be found here:
https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/download.html

Station data has been split up into ranges to make downloads more managable. You may download as much or as little as you like. To get started we reccomend just downloading a few station ranges to get an idea of how to use HadISD data with PyEarthTools. 

In [3]:
wmo_id_ranges = wmo_id_ranges # This has been defined in HadISD_data_config.ipynb

In [4]:
print(f"Downloading HadISD data for WMO range: {wmo_id_ranges}")

In [5]:
def download_wmo_range(wmo_id_range, download_dir):
    wmo_str = f"WMO_{wmo_id_range}"
    url = f"https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/data/{wmo_str}.tar.gz"
    tar_name = f"{wmo_str}.tar"
    filename = Path(download_dir) / tar_name

    head = requests.head(url, allow_redirects=True)
    remote_size = int(head.headers.get('content-length', 0))
    local_size = filename.stat().st_size if filename.exists() else 0

    if filename.exists() and local_size == remote_size:
        print(f"File already fully downloaded: {filename} ({local_size/1024**2:.2f} MB)")
    else:
        headers = {}
        mode = 'wb'
        initial_pos = 0
        if filename.exists() and local_size < remote_size:
            headers['Range'] = f'bytes={local_size}-'
            mode = 'ab'
            initial_pos = local_size
            print(f"Resuming download for {filename.name} at {local_size/1024**2:.2f} MB...")
        else:
            print(f"Starting download for {filename.name}...")

        response = requests.get(url, stream=True, headers=headers)
        total = remote_size
        with open(filename, mode) as f, tqdm(
            desc=f"Downloading {filename.name}",
            total=total,
            initial=initial_pos,
            unit='B', unit_scale=True, unit_divisor=1024
        ) as bar:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

        final_size = filename.stat().st_size
        if final_size == remote_size:
            print(f"Download complete: {filename} ({final_size/1024**2:.2f} MB)")
        else:
            print(f"Warning: Download incomplete. Local size: {final_size}, Remote size: {remote_size}")

    return filename, tar_name


### Extract Tar Files and Move to Netcdf Subfolder

In [6]:
def extract_wmo_tar(filename, tar_name, download_dir):
    extract_dir = Path(download_dir) / tar_name.replace('.tar', '')
    extract_dir.mkdir(exist_ok=True)
    extracted_files = list(extract_dir.glob('*'))
    if extracted_files:
        print(f"Extraction directory '{extract_dir}' already contains {len(extracted_files)} files. Skipping extraction.")
    elif filename.exists():
        with tarfile.open(filename, "r:gz") as tar:
            tar.extractall(path=extract_dir)
        extracted_files = list(extract_dir.glob('*'))
        if extracted_files:
            print(f"Extraction successful. {len(extracted_files)} files found in {extract_dir}.")
            filename.unlink()
            print(f"Deleted tar file: {filename}")
        else:
            print(f"Warning: No files extracted to {extract_dir}. Tar file will not be deleted.")
            raise RuntimeError("Extraction failed, tar file not deleted.")
    else:
        print("No tar file found and extraction directory is empty. Nothing to extract.")
        raise FileNotFoundError(f"Missing tar file: {filename}")
    return extract_dir

In [7]:
# Move extracted .nc files into netcdf_dir after extraction
def move_netcdf_files(extract_dir, download_dir):
    netcdf_dir = Path(download_dir) / "netcdf"
    netcdf_dir.mkdir(parents=True, exist_ok=True)
    num_files = 0
    for gz_path in extract_dir.glob('*.nc.gz'):
        nc_path = gz_path.with_suffix('')  # Remove .gz extension
        with gzip.open(gz_path, 'rb') as f_in, open(nc_path, 'wb') as f_out:
            f_out.write(f_in.read())
        gz_path.unlink()
        shutil.move(str(nc_path), netcdf_dir / nc_path.name)
        num_files += 1
    print(f"{num_files} .nc files have been extracted, cleaned up, and moved to the netcdf directory: {netcdf_dir}")

    # Delete extraction directory
    try:
        shutil.rmtree(extract_dir)
        print(f"Deleted extraction directory: {extract_dir}")
    except Exception as e:
        print(f"Could not delete extraction directory {extract_dir}: {e}")

### Idempotent Checks

In [8]:
def netcdf_files_exist_for_range(wmo_id_range, netcdf_dir):
    """Check if any .nc files for the given WMO range exist in the netcdf directory."""
    start, end = map(int, wmo_id_range.split('-'))
    nc_files = list(Path(netcdf_dir).glob("*.nc"))
    for nc_file in nc_files:
        try:
            # Extract the first 6 digits from the station part of the filename
            station_part = nc_file.stem.split('_')[-1]
            wmo_number = int(station_part.split('-')[0])
            if start <= wmo_number <= end:
                return True
        except Exception as e:
            print(f"Skipping file {nc_file.name}: {e}")
            continue
    print(f"No NetCDF files found for WMO range {wmo_id_range}.")
    return False

In [9]:
def is_tar_fully_downloaded(wmo_id_range, download_dir):
    """Check if the tar file exists and is fully downloaded (size matches remote)."""
    wmo_str = f"WMO_{wmo_id_range}"
    tar_name = f"{wmo_str}.tar"
    tar_path = Path(download_dir) / tar_name
    url = f"https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/data/{wmo_str}.tar.gz"

    if not tar_path.exists():
        return False

    # Get remote file size
    head = requests.head(url, allow_redirects=True)
    remote_size = int(head.headers.get('content-length', 0))
    local_size = tar_path.stat().st_size

    return local_size == remote_size

### Loop through each WMO range, download if necessary, extract, and move files

In [10]:
netcdf_dir = Path(download_dir) / "netcdf"
for wmo_id_range in wmo_id_ranges:
    if is_tar_fully_downloaded(wmo_id_range, download_dir):
        print(f"Tar file for {wmo_id_range} is fully downloaded. Skipping download.")
        continue

    if netcdf_files_exist_for_range(wmo_id_range, netcdf_dir):
        print(f"NetCDF files for {wmo_id_range} already exist. Skipping download and extraction.")
        continue

    filename, tar_name = download_wmo_range(wmo_id_range, download_dir)
    extract_dir = extract_wmo_tar(filename, tar_name, download_dir)
    move_netcdf_files(extract_dir, download_dir)

No NetCDF files found for WMO range 500000-549999.
Starting download for WMO_500000-549999.tar...


Download complete: /Users/joelmiller/HadISD_data/WMO_500000-549999.tar (410.71 MB)


/var/folders/v1/7wypnx5x11qgv022zwx5p3g00000gn/T/ipykernel_86163/3933807618.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


Extraction successful. 208 files found in /Users/joelmiller/HadISD_data/WMO_500000-549999.
Deleted tar file: /Users/joelmiller/HadISD_data/WMO_500000-549999.tar
208 .nc files have been extracted, cleaned up, and moved to the netcdf directory: /Users/joelmiller/HadISD_data/netcdf
Deleted extraction directory: /Users/joelmiller/HadISD_data/WMO_500000-549999
No NetCDF files found for WMO range 722000-722999.
Starting download for WMO_722000-722999.tar...


Download complete: /Users/joelmiller/HadISD_data/WMO_722000-722999.tar (1684.50 MB)
Extraction successful. 407 files found in /Users/joelmiller/HadISD_data/WMO_722000-722999.
Deleted tar file: /Users/joelmiller/HadISD_data/WMO_722000-722999.tar
407 .nc files have been extracted, cleaned up, and moved to the netcdf directory: /Users/joelmiller/HadISD_data/netcdf
Deleted extraction directory: /Users/joelmiller/HadISD_data/WMO_722000-722999
NetCDF files for 800000-849999 already exist. Skipping download and extraction.
